# 01 — Raw to Silver (quick test version)
Hardcoded values, path-based Delta writes (no Unity Catalog needed).
Reads Employees / Attendance / Performance from `raw`, cleans them, writes to `silver`.

In [0]:
STORAGE_ACCOUNT = "terrsthranalyticdevov4j3"
SILVER_SCHEMA = "silver"
IMPUTE_PERF_SCORE = False

RAW_BASE = f"abfss://raw@{STORAGE_ACCOUNT}.dfs.core.windows.net"
SILVER_BASE = f"abfss://{SILVER_SCHEMA}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

PATHS = {
    "employees": f"{RAW_BASE}/employees/",
    "attendance": f"{RAW_BASE}/attendees/",
    "performance": f"{RAW_BASE}/performance/",
}

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, trim, lower, initcap, when, coalesce, lit, expr,
    regexp_replace, avg, first
)

## 1. Extract
Performance files are tagged with `_source_file` at read time, before the union — this is what lets us later fall back to "the date used elsewhere in this same file" for rows with a missing `ReviewDate`, reliably (not relying on `input_file_name()` after transforms, which can silently break).

In [0]:
df_employees_raw = spark.read.option("header", "true").csv(PATHS["employees"])
df_attendance_raw = spark.read.option("header", "true").csv(PATHS["attendance"])

# Performance: read every .xlsx in the folder, tag each with its source file, then union
files = [f.path for f in dbutils.fs.ls(PATHS["performance"]) if f.path.endswith(".xlsx")]
dfs = []
for f in files:
    d = (
        spark.read.format("com.crealytics.spark.excel")
            .option("header", "true")
            .option("inferSchema", "false")
            .load(f)
    )
    d = d.withColumn("_source_file", lit(f))
    dfs.append(d)

df_performance_raw = dfs[0]
for d in dfs[1:]:
    df_performance_raw = df_performance_raw.unionByName(d)

print(f"Employees raw:   {df_employees_raw.count()} rows")
print(f"Attendance raw:  {df_attendance_raw.count()} rows")
print(f"Performance raw: {df_performance_raw.count()} rows")

## 2. Clean — Employees

In [0]:
DEPARTMENT_LOOKUP = {"hr": "HR", "finance": "Finance", "sales": "Sales", "it": "IT"}

df = df_employees_raw
for c in df.columns:
    df = df.withColumn(c, trim(col(c)))

df = df.withColumn("Department", when(col("Department") == "", None).otherwise(col("Department")))
dept_map = F.create_map([lit(x) for pair in DEPARTMENT_LOOKUP.items() for x in pair])
df = df.withColumn("Department", coalesce(dept_map[lower(col("Department"))], initcap(col("Department"))))
df = df.fillna({"Department": "Unknown"})

df = df.withColumn("Salary", regexp_replace(col("Salary"), "[$,]", "").cast("double"))

df = df.withColumn(
    "HireDate",
    coalesce(
        expr("try_to_date(HireDate, 'yyyy-MM-dd')"),
        expr("try_to_date(HireDate, 'MM/dd/yyyy')"),
        expr("try_to_date(HireDate, 'dd-MM-yyyy')"),
    )
)

df = df.withColumn("EmployeeID", col("EmployeeID").cast("int"))
df_employees_clean = df.dropDuplicates(["EmployeeID"])

df_employees_clean.display()

## 3. Clean — Attendance

In [0]:
MONTH_LOOKUP = {
    "01": 1, "1": 1, "jan": 1, "january": 1,
    "02": 2, "2": 2, "feb": 2, "february": 2,
    "03": 3, "3": 3, "mar": 3, "march": 3,
    "04": 4, "4": 4, "apr": 4, "april": 4,
    "05": 5, "5": 5, "may": 5,
    "06": 6, "6": 6, "jun": 6, "june": 6,
    "07": 7, "7": 7, "jul": 7, "july": 7,
    "08": 8, "8": 8, "aug": 8, "august": 8,
    "09": 9, "9": 9, "sep": 9, "september": 9,
    "10": 10, "oct": 10, "october": 10,
    "11": 11, "nov": 11, "november": 11,
    "12": 12, "dec": 12, "december": 12,
}

df = df_attendance_raw
df = df.withColumn("EmployeeID", regexp_replace(col("EmployeeID"), "(?i)EMP", "").cast("int"))

month_map = F.create_map([lit(x) for pair in MONTH_LOOKUP.items() for x in pair])
df = df.withColumn("MonthNum", month_map[lower(trim(col("Month")))])

df = df.withColumn("Year", col("Year").cast("int"))
df = df.withColumn(
    "AttendanceMonth",
    expr("try_to_date(concat(Year, '-', lpad(MonthNum, 2, '0'), '-01'), 'yyyy-MM-dd')")
)

df = df.withColumn("DaysAbsent", col("DaysAbsent").cast("int"))
df = df.withColumn("DaysAbsent", when(col("DaysAbsent") < 0, None).otherwise(col("DaysAbsent")))

df = df.withColumn("LateArrivals", coalesce(col("LateArrivals").cast("int"), lit(0)))
df = df.withColumn("DaysPresent", col("DaysPresent").cast("int"))

df_attendance_clean = df.select(
    "EmployeeID", "Year", "MonthNum", "AttendanceMonth",
    "DaysPresent", "DaysAbsent", "LateArrivals"
)

df_attendance_clean.display()

## 4. Clean — Performance

`ReviewDate`: if a row is missing its date, fall back to the date used elsewhere in
the **same source file** (review cycles are one-per-file, so this is a safe fallback —
not a guess pulled from a different month's file).

In [0]:
df = df_performance_raw
df = df.withColumn("EmployeeID", col("EmployeeID").cast("int"))

df = df.withColumn(
    "PerformanceScore",
    expr(r"try_cast(regexp_extract(PerformanceScore, '^[0-9]+\\.?[0-9]*$', 0) as double)")
)

if IMPUTE_PERF_SCORE:
    avg_score = df.select(avg("PerformanceScore")).collect()[0][0]
    df = df.fillna({"PerformanceScore": avg_score})

df = df.withColumn("ReviewDate", expr("try_to_date(ReviewDate)"))

# Fill missing ReviewDate from another row in the same source file
file_dates = (
    df.filter(col("ReviewDate").isNotNull())
      .groupBy("_source_file")
      .agg(first("ReviewDate").alias("FileReviewDate"))
)
df = df.join(file_dates, on="_source_file", how="left")
df = df.withColumn("ReviewDate", coalesce(col("ReviewDate"), col("FileReviewDate")))
df = df.drop("_source_file", "FileReviewDate")

df = df.withColumn("ManagerComment", initcap(lower(trim(col("ManagerComment")))))

df_performance_clean = df.select("EmployeeID", "ReviewDate", "PerformanceScore", "ManagerComment")

df_performance_clean.display()

## 5. Write to Silver (path-based, no catalog needed)

In [0]:
df_employees_clean.write.mode("overwrite").option("overwriteSchema", "true").format("delta").save(f"{SILVER_BASE}/employees/")
df_attendance_clean.write.mode("overwrite").option("overwriteSchema", "true").format("delta").save(f"{SILVER_BASE}/attendance/")
df_performance_clean.write.mode("overwrite").option("overwriteSchema", "true").format("delta").save(f"{SILVER_BASE}/performance/")

print("Done. Written to:")
print(f"  {SILVER_BASE}/employees/")
print(f"  {SILVER_BASE}/attendance/")
print(f"  {SILVER_BASE}/performance/")

## 6. Clean check

In [0]:
print("=" * 50)
print("EMPLOYEES")
print("=" * 50)
print("Row count:", df_employees_clean.count())
print("Duplicate EmployeeIDs:", df_employees_clean.groupBy("EmployeeID").count().filter("count > 1").count())
df_employees_clean.select([F.count(F.when(col(c).isNull(), c)).alias(c) for c in df_employees_clean.columns]).show()

print("=" * 50)
print("ATTENDANCE")
print("=" * 50)
print("Row count:", df_attendance_clean.count())
print("Null Year/Month/EmployeeID rows:", df_attendance_clean.filter(
    col("Year").isNull() | col("MonthNum").isNull() | col("EmployeeID").isNull()
).count())
print("Remaining negative DaysAbsent:", df_attendance_clean.filter(col("DaysAbsent") < 0).count())
df_attendance_clean.select([F.count(F.when(col(c).isNull(), c)).alias(c) for c in df_attendance_clean.columns]).show()

print("=" * 50)
print("PERFORMANCE")
print("=" * 50)
print("Row count:", df_performance_clean.count())
print("Null EmployeeID:", df_performance_clean.filter(col("EmployeeID").isNull()).count())
df_performance_clean.select([F.count(F.when(col(c).isNull(), c)).alias(c) for c in df_performance_clean.columns]).show()